In [1]:
using Pkg
Pkg.develop(path="C:/Users/yujie/Documents/UpgradeStockFlow/StockFlow.jl")
#Pkg.activate("C:/Users/yujie/Documents/UpgradeStockFlow/StockFlow.jl")

Pkg.instantiate()

   Resolving package versions...
  No Changes to `C:\Users\yujie\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\yujie\.julia\environments\v1.11\Manifest.toml`
Precompiling project...
    905.8 ms  ✓ JpegTurbo_jll
    891.3 ms  ✓ UpdateJulia
    817.8 ms  ✓ Libtiff_jll
   1237.9 ms  ✓ Ghostscript_jll
   1126.2 ms  ✓ GR_jll
   3329.2 ms  ✓ ForwardDiff
    902.1 ms  ✓ FastPower → FastPowerForwardDiffExt
   1129.1 ms  ✓ DifferentiationInterface → DifferentiationInterfaceForwardDiffExt
   1669.2 ms  ✓ ForwardDiff → ForwardDiffStaticArraysExt
   3608.7 ms  ✓ Latexify
   1653.0 ms  ✓ Unitful → ForwardDiffExt
   1001.1 ms  ✓ PreallocationTools → PreallocationToolsForwardDiffExt
   1416.0 ms  ✓ RecursiveArrayTools → RecursiveArrayToolsForwardDiffExt
   1048.5 ms  ✓ Latexify → SparseArraysExt
   1311.3 ms  ✓ NLSolversBase
   4717.0 ms  ✓ GR
   1833.1 ms  ✓ UnitfulLatexify
   2052.3 ms  ✓ LabelledArrays
   9015.9 ms  ✓ CommonMark
   2316.3 ms  ✓ LineSearches
   9300.8 ms  ✓ Sci

In [2]:
Pkg.status()

Status `C:\Users\yujie\.julia\environments\v1.11\Project.toml`
  [134e5e36] Catlab v0.17.1
  [f0ffcf3b] GATlab v0.2.1
  [98e50ef6] JuliaFormatter v2.1.6
  [91a5bcdd] Plots v1.40.19
  [58c4a0e8] StockFlow v0.2.4 `C:/Users/yujie/Documents/UpgradeStockFlow/StockFlow.jl`
  [770da0de] UpdateJulia v0.4.4


In [3]:
Pkg.add("LabelledArrays")
Pkg.add("OrdinaryDiffEq")

   Resolving package versions...
    Updating `C:\Users\yujie\.julia\environments\v1.11\Project.toml`
  [2ee39098] + LabelledArrays v1.16.1
  No Changes to `C:\Users\yujie\.julia\environments\v1.11\Manifest.toml`
   Resolving package versions...
    Updating `C:\Users\yujie\.julia\environments\v1.11\Project.toml`
  [1dea7af3] + OrdinaryDiffEq v6.102.1
  No Changes to `C:\Users\yujie\.julia\environments\v1.11\Manifest.toml`


In [4]:
using StockFlow

using Catlab
using Catlab.CategoricalAlgebra
using LabelledArrays
using OrdinaryDiffEq
using Plots

using Catlab.Graphics
using Catlab.Programs
using Catlab.WiringDiagrams

In [5]:
Pkg.status()

Status `C:\Users\yujie\.julia\environments\v1.11\Project.toml`
  [134e5e36] Catlab v0.17.1
  [f0ffcf3b] GATlab v0.2.1
  [98e50ef6] JuliaFormatter v2.1.6
  [2ee39098] LabelledArrays v1.16.1
  [1dea7af3] OrdinaryDiffEq v6.102.1
  [91a5bcdd] Plots v1.40.19
  [58c4a0e8] StockFlow v0.2.4 `C:/Users/yujie/Documents/UpgradeStockFlow/StockFlow.jl`
  [770da0de] UpdateJulia v0.4.4


In [6]:
Graph = StockFlow.Graph

Graph (generic function with 3 methods)

This model is re-created based on Garnett's paper:
https://journals.lww.com/stdjournal/Fulltext/2000/11000/Epidemiology_and_Control_of_Curable_Sexually.7.aspx

In [7]:
# define the function of new born
fNewBorn(u,uN,p,t)=uN.N(u,t)*p.μ
# define other functions of flows in this model, since all the other flows' functions have the 
# same form: up_stream stock * rate
# uS: name (symbol) of the upper stream stock
# r: name (symbol) of the parameter rate
function fLinear(uS,r)
    fL(u,uN,p,t) = u[uS]*p[r]
    return fL
end
# define the functions of new infectious flow
# I: the string of "A" or "Y"
function fInfectious(I)
    fIA(u,uN,p,t) = (1-p[:ϕ])*u.X*p[:cβ]*uN.NI(u,t)/uN.N(u,t)
    fIY(u,uN,p,t) = p[:ϕ]*u.X*p[:cβ]*uN.NI(u,t)/uN.N(u,t)
    if I=="A"
        return fIA
    else
        return fIY
    end
end
# define function of concatenate of two strings to symbol
# s1,s2: String
fsymbol(s1,s2)=Symbol(s1*s2)


fsymbol (generic function with 1 method)

# 1. Define the components of all the sub-models
## 1.1 The births and deaths of Stock X

In [8]:
#(stock_name=>(inflows, outflows, variables, svariables))
     ## if a stock has no inflow or no outflow, use keyword ":F_NONE"
     ## if a stock has no variables connect to, use keyword ":V_NONE"
     ## if a stock has no sum_variables connect to, use keyword ":SV_NONE"
#(flow=>variable)
#(variable=>function)
#(svariable=>variable)
     ## if sum_variable contributes to no variables, use keywork ":SVV_NONE"
openX=Open(
    StockAndFlow(
    (:X=>(:births,:deathX,:v_deathX,:N)),
    (:births=>:v_births,:deathX=>:v_deathX),
    (:v_births=>fNewBorn,:v_deathX=>fLinear(:X,:μ)),
    (:N=>:v_births)
    ),
    # feet
    foot(:X,:N,:X=>:N)
)
Graph(apex(openX))

Catlab.Graphics.Graphviz.Graph("G", true, "dot", Catlab.Graphics.Graphviz.Statement[Catlab.Graphics.Graphviz.Node("s1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "X", :shape => "square", :color => "black", :style => "filled", :fillcolor => "#9ACEEB")), Catlab.Graphics.Graphviz.Node("fs_1u", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "", :shape => "point", :color => "white")), Catlab.Graphics.Graphviz.Node("fs_2d", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "", :shape => "point", :color => "white")), Catlab.Graphics.Graphviz.Node("v1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "v_births", :shape => "plaintext", :fontcolor => "black")), Catlab.Graphics.Graphviz.Node("v2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "v_deathX", :sha

## 1.2 The SIS sub-model structure

In [9]:
#(stock_name=>(inflows, outflows, variables, svariables))
     ## if a stock has no inflow or no outflow, use keyword ":F_NONE"
     ## if a stock has no variables connect to, use keyword ":V_NONE"
     ## if a stock has no sum_variables connect to, use keyword ":SV_NONE"
#(flow=>variable)
#(variable=>function)
#(svariable=>variable)
     ## if sum_variable contributes to no variables, use keywork ":SVV_NONE"

# I: string of stock "A" or "Y"
# r: symbol of the constant parameter σ or σ′
# f: function of fInfectiousA or fInfectiousY
openSIS(I,r)=Open(
    StockAndFlow(
    (:X=>(fsymbol("recovery",I),fsymbol("newInfectious",I),fsymbol("v_newInfectious",I),:N),
     Symbol(I)=>(fsymbol("newInfectious",I),(fsymbol("death",I), fsymbol("recovery",I)),(fsymbol("v_death",I), fsymbol("v_recovery",I)),(:N,:NI))),
    (fsymbol("recovery",I)=>fsymbol("v_recovery",I),fsymbol("newInfectious",I)=>fsymbol("v_newInfectious",I),fsymbol("death",I)=>fsymbol("v_death",I)),
    (fsymbol("v_recovery",I)=>fLinear(Symbol(I),r),fsymbol("v_newInfectious",I)=>fInfectious(I),fsymbol("v_death",I)=>fLinear(Symbol(I),:μ)),
    (:N=>fsymbol("v_newInfectious",I), :NI=>fsymbol("v_newInfectious",I))
    ),
    # feet
    foot(:X,:N,:X=>:N),
    foot((),:NI,())
)


openSIS (generic function with 1 method)

In [10]:
Graph(apex(openSIS("A",:σ′)))

Catlab.Graphics.Graphviz.Graph("G", true, "dot", Catlab.Graphics.Graphviz.Statement[Catlab.Graphics.Graphviz.Node("s1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "X", :shape => "square", :color => "black", :style => "filled", :fillcolor => "#9ACEEB")), Catlab.Graphics.Graphviz.Node("s2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "A", :shape => "square", :color => "black", :style => "filled", :fillcolor => "#9ACEEB")), Catlab.Graphics.Graphviz.Node("fs_3d", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "", :shape => "point", :color => "white")), Catlab.Graphics.Graphviz.Node("v1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "v_recoveryA", :shape => "plaintext", :fontcolor => "black")), Catlab.Graphics.Graphviz.Node("v2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Grap

In [11]:
Graph(apex(openSIS("Y",:σ)))

Catlab.Graphics.Graphviz.Graph("G", true, "dot", Catlab.Graphics.Graphviz.Statement[Catlab.Graphics.Graphviz.Node("s1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "X", :shape => "square", :color => "black", :style => "filled", :fillcolor => "#9ACEEB")), Catlab.Graphics.Graphviz.Node("s2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "Y", :shape => "square", :color => "black", :style => "filled", :fillcolor => "#9ACEEB")), Catlab.Graphics.Graphviz.Node("fs_3d", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "", :shape => "point", :color => "white")), Catlab.Graphics.Graphviz.Node("v1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:label => "v_recoveryY", :shape => "plaintext", :fontcolor => "black")), Catlab.Graphics.Graphviz.Node("v2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Grap

# 2 Compose
## 2.1 Define composition rule

In [12]:
# define the UWD-algebra
uwd = @relation (XN,NI) begin
    X(XN)
    XA(XN,NI)
    XY(XN,NI)
end;
display_uwd(uwd)

Catlab.Graphics.Graphviz.Graph("G", false, "neato", Catlab.Graphics.Graphviz.Statement[Catlab.Graphics.Graphviz.Node("n1", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:id => "box1", :label => "X")), Catlab.Graphics.Graphviz.Node("n2", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:id => "box2", :label => "XA")), Catlab.Graphics.Graphviz.Node("n3", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:id => "box3", :label => "XY")), Catlab.Graphics.Graphviz.Node("n4", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:height => "0", :id => "outer1", :label => "", :margin => "0", :shape => "none", :style => "invis", :width => "0")), Catlab.Graphics.Graphviz.Node("n5", OrderedCollections.OrderedDict{Symbol, Union{String, Catlab.Graphics.Graphviz.Html}}(:height => "0", :id => "outer2", :label => "", :margin => "0", :shape => "none", :style => "i

In [13]:
const C

Base.Meta.ParseError: ParseError:
# Error @ c:\Users\yujie\Documents\UpgradeStockFlow\StockFlow.jl\examples\full_fledged_schema_examples\composition\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X32sZmlsZQ==.jl:1:1
const C
└─────┘ ── expected assignment after `const`

In [14]:
XAY_model = oapply(uwd,Dict(
    :X=>openX,
    :XA=>openSIS("A",:σ′),
    :XY=>openSIS("Y",:σ)
    )) |> apex
Graph(XAY_model)

MethodError: MethodError: Cannot `convert` an object of type 
  Right{typeof(fNewBorn)} to an object of type 
  Union{Left{Union{}}, Right{Any}}
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  convert(::Type{T}, !Matched::T) where T
   @ Base Base.jl:126


In [15]:
Graph(XAY_model,type="SF")

UndefVarError: UndefVarError: `XAY_model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# 3. Solve ODEs

In [16]:
# define values of constant parameters
# unit: year
p = LVector(
    cβ=15*0.7, μ=1/15,ϕ=(0.95+0.4)/2,σ=12/4,σ′=12/4
)
# define initial values for stocks
u0 = LVector(
    X=990, A=5, Y=5
)

3-element LArray{Int64, 1, Vector{Int64}, (:X, :A, :Y)}:
 :X => 990
 :A => 5
 :Y => 5

In [17]:
# results are tested the same as the Anylogic model
prob_XAY = ODEProblem(vectorfield(XAY_model),u0,(0.0,2.0),p);
sol = solve(prob_XAY,Tsit5(),abstol=1e-8);
plot(sol)

UndefVarError: UndefVarError: `XAY_model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
# to have the figures plotted fix to the wider of the cells
HTML("""
<style>
.output_svg div{
  width: 100% !important;
  height: 100% !important;
}
</style>
""")


HTML{String}("<style>\n.output_svg div{\n  width: 100% !important;\n  height: 100% !important;\n}\n</style>\n")